## DSPy Ollama Qwen2 Information Extraction

#### Load in Python Libraries

In [68]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rich import print
import pandas as pd
import ast

from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch

from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2','rougeL'], use_stemmer=True)
import json

from data.train_examples import train_example_list
from data.valid_examples import dev_example_list
from data.test_example import test_examples_list

#### Helper Functions

In [69]:
def validate_ans(example, pred, trace = None):

    gold = re.sub(r'\n|\s+ ', '',dict(example)['answer']).lower()
    print(gold)

    prediction = pred.answer.lower()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Train, Dev and Test

In [70]:
train_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet1.csv')

dev_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/50examples_for_DSPy_withJson.csv')

test_examples = pd.read_csv("/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet2.csv", header = None)

#### Set LLM Qwen2

In [71]:
llm = dspy.OllamaLocal(model='qwen2:latest', max_tokens = 4000, temperature=0.0)
dspy.settings.configure(lm=llm)

#### Create DSPy Signature and Module

In [72]:
class GenerateAnswer(dspy.Signature):
    """Extract information from a job posting and return the output in a json format if you don't know answer Not Specified. Should be key-value with output as dictionary."""

    context = dspy.InputField(desc="contain relevant facts")
    question = dspy.InputField(desc="unique possible questions")
    answer = dspy.OutputField(desc="key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words")

class QUESTIONANSWER(dspy.Module):
    def __init__(self,question):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens=400)
        self.question=question

    def forward(self, context):
        context = context.replace('\n', ' ').replace('“', '"').replace('”', '"')
        context = normalize(context)
        question=self.question
        pred = self.generate_answer(context=context, question=question)
        pred = re.sub(r"```\n|```", "",pred.answer)
        return dspy.Prediction(context=context,answer=pred)

In [73]:
uncompiled_fs=QUESTIONANSWER('''
                            "position_title" : What is the title of this position?
                            "location" : Where is this position located, including city, state and zip code?
                            "work_arrange" : What is the work arrangement for this position, remote, hybrid, or on-site?
                            "experience" : what are years of experience required for this position?
                            "employment_type" : What is the employment type, full time, part time, or internship?
                            "pay" : What is the pay for this position?
                            "degree" : What is required degree?
                            "certifications" : What certifications or qualifications are required?
                            "required_skills" : What are required skills?
                              ''')

#### Test Uncompiled DSPy 

In [74]:
print(train_examples['body'][2])

GIS Technician Dobson Fiber 14101 Wireless Way, Oklahoma City, OK 73134 Full-time Dobson Fiber 16 reviews Read what
people are saying about working here. Job details No matching Job Type Full-time Shift and Schedule 8 hour shift 
SUMMARY: Position will be responsible for updating and maintaining GIS databases, creating webmaps, and assisting 
engineers with GIS data manipulation. GIS Technicians will work in both ArcGIS Pro and AGOL to create, modify, and 
publish materials to assist internal and external stakeholders. This position will be responsible for interpreting,
communicating, and visualizing spatial data as needed. ESSENTIAL FUNCTIONS: Create, update, and maintain geospatial
data in ArcGIS software applications and databases. Publish/Edit Content for ArcGIS Online and Enterprise Portal. 
Web Map/App Creation and Maintenance. Collaborate with project leaders and engineers on GIS-related updates from 
construction projects. Partner with subject matter experts to resolve discrepancies, improve data quality, and 
provide solutions to problems or issues via GIS. Write, update, and utilize ArcPy to automate repetitive tasks. 
Support department GIS users with spatial reports and analysis, data editing, QA/QC, workflow management, and 
application training. REQUIRED KNOWLEDGE, SKILLS, ABILITIES AND ATTRIBUTES: Experience with ESRI ArcGIS Desktop or 
ArcPro software including ESRI Apps like Field Maps/ArcGIS Online (AGOL)/ESRI Portal and Microsoft Office suite 
(Excel, Word, PowerPoint) Required. File and generate reports; clerical skills; ability to understand and carry out
complex oral and written directions; ability to work under pressure with good organizational skills; ability to 
work in a team environment; initiative in recognizing need for improvement in, or adaptation of, existing systems 
and effecting changes in them; in tracking down missing and misfiled items; accuracy in filing; physical condition 
commensurate with the demands of the position. MINIMUM ACCEPTABLE TRAINING AND EXPERIENCE: Bachelors degree in a 
related field and/or a minimum of three to five years GIS experience. ADDITIONAL SKILLS REQUIRED: Strong data base,
software and operating systems background Strong Analytical Skills Very meticulous Good problem-solving skills Good
communication skills Must be able to prioritize projects and multi-task Job Type: Full-time Benefits: 401(k) 401(k)
matching Dental insurance Employee assistance program Flexible spending account Health insurance Health savings 
account Life insurance Paid time off Retirement plan Vision insurance Schedule: 8 hour shift Ability to 
commute/relocate: Oklahoma City, OK 73134: Reliably commute or planning to relocate before starting work (Required)
Experience: GIS: 3 years (Preferred) Work Location: In person If you require alternative methods of application or 
screening, you must approach the employer directly to request this as Indeed is not responsible for the employer's 
application process.

In [75]:
with dspy.context(lm = llm):
    pred = uncompiled_fs(context = dev_examples['body'][2])
    print(pred.answer)

{
  "position_title": "Pharmacy Technician",
  "location": "San Quentin, CA, 94964",
  "work_arrange": "On-site",
  "experience": "1 year",
  "employment_type": "Contract",
  "pay": "$18 - $19 an hour",
  "degree": "High School Diploma or GED",
  "certifications": "Pharmacy Technician License, Basic Life Support, COVID certification",
  "required_skills": "Customer Service Experience"
}

#### Create Train, Dev, Test Split

In [76]:
print(len(dev_example_list) , len(train_example_list), len(test_examples_list))
train_results = train_example_list
train_contents = list(train_examples['body'])

dev_results = dev_example_list
dev_contents = list(dev_examples.loc[:20,'body'])

test_results = test_examples_list
test_contents = list(test_examples[1])

train_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(train_contents, train_results)]
dev_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(dev_contents, dev_results)]
test_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(test_contents, test_results)]

trainset=train_examples_list
devset=dev_examples_list
testset = test_examples_list

trainset = [x.with_inputs('context') for x in trainset]
devset = [x.with_inputs('context') for x in devset]
testset = [x.with_inputs('context') for x in testset]

21 20 10

#### Test on Test set

In [77]:
answ = testset[0]
print(answ.answer)

{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", "work_arrangement": 
"Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": "$120K/year", "degree": "A 
BS in the Engineering field", "certifications": "CRM (salesforce.com), RFQ experience and price quotes to the DOD",
"required_skills": "Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps"}

In [78]:
with dspy.context(lm=llm):
    pred = uncompiled_fs(context=testset[0].context)
    print(pred.answer)


{
  "position_title": "Federal Sales Engineer",
  "location": "Walpole, MA, USA",
  "work_arrange": "Hybrid remote",
  "experience": "Experience in the development and selling of power electronics or similar hardware",
  "employment_type": "Full time",
  "pay": "$120k/year",
  "degree": "BS in Engineering field",
  "certifications": "CRM (Salesforce.com), RFQ experience, and price quotes to the DoD preferred; working knowledge
of ERP (Epicor is preferred)",
  "required_skills": "Technical sales experience, experience working with outside sales reps"
}

In [79]:
validate_ans(answ, pred)

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certifications": "crm (salesforce.com), rfq experience and price quotes to the dod",
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{
  "position_title": "federal sales engineer",
  "location": "walpole, ma, usa",
  "work_arrange": "hybrid remote",
  "experience": "experience in the development and selling of power electronics or similar hardware",
  "employment_type": "full time",
  "pay": "$120k/year",
  "degree": "bs in engineering field",
  "certifications": "crm (salesforce.com), rfq experience, and price quotes to the dod preferred; working knowledge
of erp (epicor is preferred)",
  "required_skills": "technical sales experience, experience working with outside sales reps"
}

0.6473572037510656

0.6473572037510656

#### Compile Information extraction Model

In [80]:
teleprompter = BootstrapFewShot(metric=validate_ans)
compiled_info_extract = teleprompter.compile(uncompiled_fs, trainset=trainset)

# teleprompter = BootstrapFewShotWithRandomSearch(metric=validate_ans, max_labeled_demos=16, max_rounds=1,  max_errors = 5, stop_at_score=0.50) 
# compiled = teleprompter.compile(uncompiled_fs, trainset=trainset, valset = devset)

  0%|          | 0/20 [00:00<?, ?it/s]

{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

the position is likely titled "drilling rig operator" or a similar role within drilling operations or oilfield 
services. the employment type is full-time, as indicated by the presence of benefits such as medical insurance and 
401(k) plans. the pay for this position would be determined based on market rates and could be expressed in terms 
of "$ / hour". a high school diploma, ged, or equivalent is required for the degree level needed. certifications 
include a cdl b license for driving crew trucks, and candidates must meet all qualifications defined in the motor 
vehicle policy if applicable. required skills encompass effective communication, teamwork ability, fast-paced work 
environment management, problem-solving, basic mechanical knowledge, and understanding of safety protocols specific
to an oilfield setting.

reasoning:
1. **position title**: the context provided suggests a role closely related to drilling operations or oilfield 
services, leading to the title "drilling rig operator" as a plausible fit.
2. **employment type**: full-time employment is inferred from the inclusion of benefits like medical insurance and 
401(k) plans, which are typically associated with full-time positions offering comprehensive employee benefits.
3. **pay**: the "$ / hour" notation suggests that the actual hourly wage would be determined based on market 
conditions or company policy, indicating a variable but quantifiable compensation structure.
4. **degree**: a high school diploma or equivalent is specified as the minimum educational requirement, aligning 
with roles requiring technical skills without necessarily demanding advanced academic qualifications.
5. **certifications**: the mention of a cdl b license indicates that candidates must possess this specific driving 
certification to operate vehicles used in drilling operations, suggesting additional safety and operational 
certifications might also be required based on industry standards or company policies.
6. **required skills**: the skills listed are essential for performing the duties effectively: communication 
(verbal and written), teamwork, adaptability to fast-paced environments, problem-solving capabilities, basic 
mechanical knowledge, and adherence to stringent safety protocols in an oilfield context.

this breakdown provides a comprehensive understanding of the position based on the information provided.

0.08660738225042452

  5%|▌         | 1/20 [00:25<07:57, 25.15s/it]

...

 15%|█▌        | 3/20 [00:56<05:08, 18.13s/it]

{"position_title": "graphic designer","location": "goochland, va","work_arrangement": "hybrid, with two in-office 
days per week","experience": "minimum 5 years design and publications experience","employment_type": 
"part-time","pay": "not specified","degree": "college degree in graphic design, visual arts, or related 
field","certification": "not specified","required_skills": "proficiency with indesign, photoshop, illustrator, 
working knowledge of constant contact, strong organizational skills, excellent oral/written communication and 
client-relations skills, ability to work under pressure, working knowledge of ap style, 35mm and digital 
photography skills, mac environment"}

the required skills for the position described include:

1. **proficiency in design software**: this includes software such as adobe indesign, photoshop, illustrator, and 
other relevant design tools necessary for creating newsletters, publications, and other visual content.

2. **strong organizational skills**: the ability to manage multiple projects simultaneously, prioritize tasks 
effectively, and maintain a high level of organization is crucial.

3. **excellent communication skills**: both written and verbal communication skills are essential for conveying 
ideas clearly and collaborating with team members and stakeholders.

4. **ability to work under pressure**: the role may involve tight deadlines and the need to produce quality work 
quickly, requiring strong time management and stress management abilities.

5. **working knowledge of ap style**: understanding and applying associated press (ap) style guidelines ensures 
consistency in writing across publications.

6. **direct mail experience**: familiarity with direct mail campaigns can be beneficial for roles that involve 
marketing communications or fundraising efforts.

7. **attention to detail**: precision is critical in design work, ensuring that all elements of a project are 
accurate and polished.

8. **time-management skills**: efficiently allocating time across various tasks ensures projects are completed on 
schedule without compromising quality.

9. **positive attitude**: a proactive approach and a positive outlook contribute to a productive work environment 
and successful outcomes.

10. **team player qualities**: collaboration with colleagues is essential for the smooth operation of any 
team-based role, requiring good interpersonal skills and the ability to work well in a group setting.

11. **high level of initiative and self-motivation**: the capacity to take initiative on projects and drive oneself
towards goals without constant supervision fosters personal growth and contributes to the success of the 
organization.

12. **desire for ongoing education and training**: continuous learning is encouraged, as staying updated with 
industry trends and acquiring new skills enhances professional capabilities and job performance.

these skills collectively ensure that a candidate can effectively manage communications projects, produce 
high-quality content, and contribute positively to the team's objectives within the va farm bureau or similar 
organization.

0.07854231834837787

 20%|██        | 4/20 [01:26<05:45, 21.61s/it]


In [81]:
with dspy.context(lm=llm):
    pred = compiled_info_extract(context=testset[0].context)
    print(pred.answer)

Position Title: Senior Inside Sales Rep/Sales Engineer

Location: Walpole, MA, USA

Work Arrange: Hybrid remote (on-site and remote)

Experience: Not explicitly stated but significant experience is required based on responsibilities and 
requirements.

Employment Type: Full time

Pay: Competitive salary (base + commission) with additional benefits including paid vacation, federal holidays, 
401(k), medical and dental insurance, individual performance bonus.

Degree: Bachelor's degree in an engineering field

Certifications/Qualifications: Experience working with outside sales reps, CRM (Salesforce.com), RFQ experience and
price quotes to the DoD preferred. Familiarity with ERP systems like Epicor is preferred. Knowledge of power 
electronics and sales experience in military and industrial sectors considered a plus.

Required Skills:
- Technical sales experience
- Ability to work with outside sales reps
- Proficiency with CRM systems (Salesforce.com)
- RFQ experience and price quotes to the DoD
- Familiarity with ERP systems (Epicor preferred)
- Knowledge of power electronics
- Problem-solving skills related to installed equipment

In [82]:
validate_ans(answ, pred)

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certifications": "crm (salesforce.com), rfq experience and price quotes to the dod",
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

position title: senior inside sales rep/sales engineer

location: walpole, ma, usa

work arrange: hybrid remote (on-site and remote)

experience: not explicitly stated but significant experience is required based on responsibilities and 
requirements.

employment type: full time

pay: competitive salary (base + commission) with additional benefits including paid vacation, federal holidays, 
401(k), medical and dental insurance, individual performance bonus.

degree: bachelor's degree in an engineering field

certifications/qualifications: experience working with outside sales reps, crm (salesforce.com), rfq experience and
price quotes to the dod preferred. familiarity with erp systems like epicor is preferred. knowledge of power 
electronics and sales experience in military and industrial sectors considered a plus.

required skills:
- technical sales experience
- ability to work with outside sales reps
- proficiency with crm systems (salesforce.com)
- rfq experience and price quotes to the dod
- familiarity with erp systems (epicor preferred)
- knowledge of power electronics
- problem-solving skills related to installed equipment

0.2907181054239878

0.2907181054239878

In [83]:
compiled_info_extract.save("compiled_v2.json")

#### Evaluate Uncompiled vs Compiled

In [84]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10)

prev_score=evaluation(uncompiled_fs, metric=validate_ans)

improved_score=evaluation(compiled_info_extract, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certifications": "crm (salesforce.com), rfq experience and price quotes to the dod",
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{
  "position_title": "federal sales engineer",
  "location": "walpole, ma, usa",
  "work_arrange": "hybrid remote",
  "experience": "experience in the development and selling of power electronics or similar hardware",
  "employment_type": "full time",
  "pay": "$120k/year",
  "degree": "bs in engineering field",
  "certifications": "crm (salesforce.com), rfq experience, and price quotes to the dod preferred; working knowledge
of erp (epicor is preferred)",
  "required_skills": "technical sales experience, experience working with outside sales reps"
}

0.6473572037510656

Average Metric: 0.6473572037510656 / 1  (64.7):  10%|█         | 1/10 [00:10<01:31, 10.15s/it]

...

{"position_title": "cryptocurrency / fx trader - entry level", "location": "not specified", "work_arrangement": 
"remote", "experience": "no prior experience required", "employment_type": "full-time or part-time", "pay": 
"results-based commissions and performance bonuses", "degree": "bachelor's degree in finance, economics, or related
field preferred", "certifications": "none specified", "required_skills": "strong analytical skills, quick 
decision-making"}

position title: contract trader (or forex/cryptocurrency trader)

location: anywhere globally with high-speed internet availability.

work arrange: remote work arrangement is offered, allowing individuals to work from anywhere as long as they have 
access to a high-speed internet connection.

experience: while not explicitly stated in terms of years, the position suggests that some level of experience is 
expected or preferred. experienced traders are mentioned who can earn over $100,000 annually and have unlimited 
earnings potential.

employment type: contract-based employment type is provided, indicating flexibility in hours and potentially 
transitioning from part-time to full-time work based on performance and experience.

pay: the pay for this position is described as potentially earning over $100,000 annually with unlimited earnings 
potential. this suggests that compensation can vary significantly based on the individual's performance and skills.

degree: a finance, economics, or related field degree is preferred but not required. this indicates that formal 
education in these areas might be advantageous for candidates applying to this position.

certifications: no specific certifications are mentioned as requirements for this position. however, experience and
knowledge in financial markets would likely be beneficial.

required skills: the skills required are not detailed in the provided information; however, given the nature of 
trading forex and cryptocurrencies, skills such as market analysis, risk management, technical and fundamental 
analysis, and proficiency with trading platforms or software might be expected for this role.

0.10641543454742156

Average Metric: 3.293092935415386 / 10  (32.9): 100%|██████████| 10/10 [03:30<00:00, 21.09s/it]


,example_context,example_answer,pred_context,pred_answer,validate_ans
0,"Federal Sales Engineer - Tech & ISR Experience - Hybrid Remote - Walpole MA Advanced Recruiting Solutions Walpole, MA Depends on Experience Full Time Work...","{""position_title"": ""Senior Inside Sales Rep/Sales Engineer"", ""location"": ""Walpole, MA"", ""work_arrangement"": ""Hybrid"", ""experience"": ""Depends on Experience"", ""employment_type"": ""Full Time"", ""pay"": ""$120K/year"", ""degree"": ""A BS in the...","federal sales engineer - tech & isr experience - hybrid remote - walpole ma advanced recruiting solutions walpole, ma depends on experience full time work...","Position Title: Senior Inside Sales Rep/Sales Engineer Location: Walpole, MA, USA Work Arrange: Hybrid remote (on-site and remote) Experience: Not explicitly stated but significant experience...",✔️ [0.2907181054239878]
1,"Position Description Penetration Tester Location Washington, DC Req # 12763 # of openings 2 ECS is seeking a Penetration Tester to work in our Washington,...","{""position_title"": ""Penetration Tester"", ""location"": ""Washington, DC"", ""work_arrangement"": ""On-site"", ""experience"": ""10+ years of Penetration Testing experience"", ""employment_type"": ""Full time"", ""pay"": ""Not specified"", ""degree"": ""Bachelors Degree in...","position description penetration tester location washington, dc req # 12763 # of openings 2 ecs is seeking a penetration tester to work in our washington,...","Position Title: Penetration Tester Location: Washington, DC Work Arrange: On-site Experience: 10 years of penetration testing experience required Employment Type: Full-time (assumed based on context)...",✔️ [0.45359589041095894]
2,NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility Inc. compensation: HOURLY...,"{""position_title"": ""NURSES - RNs or LPNs"", ""location"": ""Sudbury, MA 01776"", ""work_arrangement"": ""on-site"", ""experience"": ""Minimum of 1 year Long term care experience/SNF experience preferred"", ""employment_type"": ""full-time"",...",nurses - rns & lpns in snf - sign on bonus - child daycare on site (sudbury) sudbury pines extended care facility inc. compensation: hourly...,"The position title for this role is ""Nurses - RNS or LPNs"". The location specified is Sudbury Pines Extended Care Facility, located at Sudbury, MA...",✔️ [0.1598508169547715]
3,Planner IV - Transportation Planner Job Details Apply Print Share This listing closes on 7/17/2023 at 11:59 PM Pacific Time (US & Canada); Tijuana. Salary...,"{""position_title"": ""Planner IV - Transportation Planner"", ""location"": ""Yakima, WA, 98901"", ""work_arrangement"": ""On-site"", ""experience"": ""5 years of increasingly responsible professional experience"", ""employment_type"": ""Full-Time"", ""pay"": ""$39.84 -...",planner iv - transportation planner job details apply print share this listing closes on 7/17/2023 at 11:59 pm pacific time (us & canada); tijuana. salary...,"Position Title: Planner IV - Transportation Planner Location: Yakima, Washington, 98901 Work Arrange: On-site/full-time Experience: Five years of professional experience Employment Type: Full time Pay:...",✔️ [0.41702127659574467]
4,"Associate Attorney Juan Ramos Law Group, PLLC McAllen, TX Job Details Full-time From $50,000 a year 1 day ago Qualifications Spanish Law Doctoral degree Criminal...","{""position_title"": ""Associate Attorney"", ""location"": ""McAllen, TX"", ""work_arrangement"": ""on-site"", ""experience"": ""None specified."", ""employment_type"": ""full-time"", ""pay"": ""$50,000 a year"", ""degree"": ""Law doctoral degree"", ""certifications"": ""Admission to the...","associate attorney juan ramos law group, pllc mcallen, tx job details full-time from $50,000 a year 1 day ago qualifications spanish law doctoral degree criminal...","Position Title: Associate Attorney Location: McAllen, Texas Work Arrange: On-site Experience: Newly graduated or wi

In [85]:
prev_score, improved_score

(64.81, 32.93)

In [86]:
with dspy.context(lm=llm):
    pred = compiled_info_extract(devset[1].context)
    print(pred.answer)

The required components for this position are as follows:

1. **Employment Type**: Full-time.
2. **Pay**: The pay range is from $32,703 to $45,000 per year.
3. **Degree**: A bachelor's degree in human services or business fields such as human resources, marketing, 
business administration, healthcare management, or public administration is required. Alternatively, an associate 
degree in a human services or business field from an appropriately accredited institution and four years of 
directly related work experience can be accepted.
4. **Certifications**: No specific certifications are mentioned; however, familiarity with the Workforce Innovation
and Opportunity Act (WIOA) or knowledge of vocational rehabilitation processes might be beneficial but not 
explicitly required.
5. **Required Skills**:
   - Knowledge of vocational rehabilitation services.
   - Ability to work with diverse populations.
   - Strong communication skills.
   - Proficiency in using technology for data management, reporting, and communication.
   - Understanding of employment laws and regulations related to disability rights.

These details provide a comprehensive overview of the requirements for this position within the Division of 
Vocational Rehabilitation Services (DVRS) under the Department of Health and Human Services (DHHS).

In [87]:
compiled_info_extract.save("compiled_evaluate.json")


In [88]:
len(llm.history)

56